# Decision Trees for Binary Classification

This notebook fits a decision tree on a binary classification dataset to
empirically test how different hyperparameters help the algorithm
generalize better on data that was not used to train the model.

---

Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score, precision_score, recall_score, roc_curve, roc_auc_score
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.tree import DecisionTreeClassifier

## Data

Read dataset

In [ ]:
URL_DATA = 'https://raw.githubusercontent.com/nmiuddin/UCI-Heart-Disease-Dataset/refs/heads/master/data/heart-disease-UCI.csv'
df = pd.read_csv(URL_DATA)

Explore dataset

In [ ]:
print('Observed positive rate:', round(df['target'].mean(), 2))
df.head()

Split dataset

- `train` is data that we will use to train the classifier.
- `test` is data that we will **NOT** show the classifier during the
training
process. Imagine it's data from the future that was not available when you were
training the model. We will use this to evaluate the model's performance.

In [ ]:
# Separate features from target
X = df.drop('target', axis=1)
y = df['target']

# Split into train and test datasets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Naive Decision Tree Classifier

Declare classifier

In [ ]:
# Init classifier
clf = DecisionTreeClassifier(
    criterion='gini', splitter='best', max_depth=None, min_samples_split=2,
    max_features=None, random_state=42
)

# Fit the instance we declared earlier
clf.fit(X_train, y_train)

Let's use the _fitted_ classifier instance we just created to generate predictions.

In [ ]:
# Predict on the same data used for training
pred_train = clf.predict(X_train)  # Predicted classes
pred_train_prob = clf.predict_proba(X_train)  # Predicted probabilities

# Predict on testing data (never before seen by the model)
pred_test = clf.predict(X_test)  # Predicted classes
pred_test_prob = clf.predict_proba(X_test)  # Predicted probabilities

Let's evaluate the model's performance

In [ ]:
# Accuracy
train_acc = (y_train == pred_train).mean()
test_acc = (y_test == pred_test).mean()

# Precision (share of all positive predictions that were accurate)
train_prc = precision_score(y_train, pred_train)
test_prc = precision_score(y_test, pred_test)

# Recall (share of all positive instances detected)
train_rec = recall_score(y_train, pred_train)
test_rec = recall_score(y_train, pred_train)

# F1 (harmonic mean of precission and recall)
train_f1 = f1_score(y_train, pred_train)
test_f1 = f1_score(y_test, pred_test)

As you can see, there is a significant performance drop.

Ideally, we want a model that generalizes well when evaluated on new data. In other
words, we want a model that has really good performance on both training and testing
data.

We should therefore use the training and testing performance metrics to estimate the
model's variance and pick a _good model_ that has low bias **and** lo

In [ ]:
# Pre-formatted string
_msg = """{} Evaluation
- Train: {:.3f},
- Test: {:.3f}
"""

# Print summary
print(
    _msg.format('F1', train_f1, test_f1)
)

## K-Fold Cross-Validation

K-fold cross-validation is a technique used to estimate how well a machine learning
model generalizes to an independent dataset. The core idea is to partition the training
dataset into _K_ equally sized subsets (_folds_). The model is then trained _K_ times,
and in each iteration, one fold is reserved as the validation set (for testing), while
the remaining _K-1_ folds are used for training. This process allows us to sample _K_
instances of the testing error, which is much better than simply having one observation.

The main benefit of K-fold cross-validation is that it provides a more reliable
estimate of the model's performance compared to a single train-test split, especially
when dealing with smaller datasets. By averaging the performance metrics across all _K_
iterations, it helps to reduce the variance of the performance estimate, giving a
better indication of how the model will perform on unseen data and aiding in the
selection of optimal hyperparameters.

In [ ]:
# Declare grid (hyperparameter configurations)
grid = {
    'max_depth': np.arange(1, 9),
    'min_samples_split': [100, 50, 25, 2],
    'max_features': [None, 4, 3]
}  # Total of 96 different configurations (8 * 4 * 3)

# Init grid search object with K=4
search = GridSearchCV(
    estimator=clf, param_grid=grid, scoring='f1', n_jobs=-1, cv=4,
    return_train_score=True, refit=True  # Retrain the best model on 100% of Train
)

# Fit grid search object
search.fit(X_train, y_train)

We can call `search`'s `cv_results_` attribute to view the results of each
configuration.

In [ ]:
# Store restuls
search_results = pd.DataFrame(search.cv_results_)

# View top 5 models
search_results[[
    'param_max_depth', 'param_max_features', 'param_min_samples_split',
    'mean_train_score', 'mean_test_score', 'rank_test_score'
]].sort_values('rank_test_score', ascending=True).head()

There are many ways to pick the best model. `GridSearchCV` automatically picks the
configuration with the highest average score on test. Another option would be to pick
the set of hyperparameters that have the smallest difference between train and test
performance subject to a minimum test score.

## Receiver Operating Characteristic Curve

## Receiver Operating Characteristic (ROC) Curve

The Receiver Operating Characteristic (ROC) curve is used to see a binary classifier's
ability to discriminate as its threshold varies. It plots two parameters:

*   **True Positive Rate (TPR)**: Also known as _sensitivity_. It's calculated as
$\frac{\text{True Positives}}{\text{True Positives} + \text{False Negatives}}$.
*   **False Positive Rate (FPR)**: It's calculated as
$\frac{\text{False Positives}}{\text{False Positives} + \text{True Negatives}}$.

The **Area Under the ROC Curve (AUC)** is a measure of the ability of a classifier to
distinguish between classes and is used as a summary of the ROC curve. The higher the
AUC, the better the model. An AUC of 0.5 suggests no discrimination (equivalent to
random guessing), while an AUC of 1.0 represents a perfect classifier.

In [ ]:
# Overwrite clf with best classifier
clf = search.best_estimator_

# Get predicted probabilities for the positive class (class 1) **ON TESTING DATASET**
y_pred_prob = clf.predict_proba(X_test)[:, 1]

# Calculate FPR, TPR, and thresholds
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)

# Calculate AUC score
auc = roc_auc_score(y_test, y_pred_prob)

# Plot the ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {auc:.2f})')
plt.plot([0, 1], [0, 1], ls='--', label='Random Classifier', color='gray')
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

# Ensemble Methods

## Random Forests

As you've learned, a shallow decision tree is better than a deep one because the latter
is likely to overfit.

If a shallow tree is so good, then what would happen if we fit many shallow trees and
have them vote to come to a consensus of what class an observation is most likely to
belong to?

That's decision trees in a nutshell: a collection of many different instantiations of
individual decision trees that share the same hyperparameters.

In [ ]:
# Extract params of best tree classifier
dt_params = search.best_estimator_.get_params()

# Init a forest of many of those trees
rf = RandomForestClassifier(
    n_estimators=100,
    criterion=dt_params['criterion'],
    max_depth=dt_params['max_depth'],
    min_samples_split=dt_params['min_samples_split'],
    max_features=dt_params['max_features'],
    random_state=42  # Added random_state for reproducibility
)

# Fit
rf.fit(X_train, y_train)

# Calculate F1 scores for Decision Tree
dt_f1_train = f1_score(y_train, clf.predict(X_train))
dt_f1_test = f1_score(y_test, clf.predict(X_test))

# Calculate F1 scores for Random Forest
rf_f1_train = f1_score(y_train, rf.predict(X_train))
rf_f1_test = f1_score(y_test, rf.predict(X_test))

print(f"Decision Tree (Train F1): {dt_f1_train:.3f}")
print(f"Decision Tree (Test F1): {dt_f1_test:.3f}")
print(f"Random Forest (Train F1): {rf_f1_train:.3f}")
print(f"Random Forest (Test F1): {rf_f1_test:.3f}")

An ensemble of many instances of the best single tree results in better predictions
than just one of those trees.

We can do even better by training many shallower trees than the one that won in the
single decision tree case.

In [ ]:
# Declare grid for Random Forest (shallower parameters)
grid_rf = {
    'n_estimators': [50, 100, 150, 700],
    'max_depth': np.arange(1, dt_params['max_depth']),  # Even shallower than DT
    'min_samples_split': [50, 25],
    'max_features': ['sqrt', None]
}

# Initialize a RandomForestClassifier estimator
rf = RandomForestClassifier(random_state=42)

# Init grid search object for Random Forest with K=4
search_rf = GridSearchCV(
    estimator=rf,
    param_grid=grid_rf,
    scoring='f1',
    n_jobs=-1,
    cv=4,
    return_train_score=True, refit=True  # Retrain winner on 100% of Train
)

# Fit grid search object
search_rf.fit(X_train, y_train)

# Store results
search_rf_results = pd.DataFrame(search_rf.cv_results_)

# View top 5 models
print('Top 5 Random Forest models:')
search_rf_results[[
    'param_n_estimators', 'param_max_depth', 'param_min_samples_split',
    'param_max_features', 'mean_train_score', 'mean_test_score', 'rank_test_score'
]].sort_values('rank_test_score', ascending=True).head()

We can easily beat the best decision tree's F1 with a Random Forest ensemble!

In [ ]:
# Get the best Decision Tree and Random Forest estimators
best_dt_clf = search.best_estimator_
best_rf_clf = search_rf.best_estimator_

# Calculate F1 scores for the best Decision Tree
best_dt_f1_train = f1_score(y_train, best_dt_clf.predict(X_train))
best_dt_f1_test = f1_score(y_test, best_dt_clf.predict(X_test))

# Calculate F1 scores for the best Random Forest
best_rf_f1_train = f1_score(y_train, best_rf_clf.predict(X_train))
best_rf_f1_test = f1_score(y_test, best_rf_clf.predict(X_test))

print(f"Best Decision Tree (Train F1): {best_dt_f1_train:.3f}")
print(f"Best Decision Tree (Test F1): {best_dt_f1_test:.3f}")
print(f"Best Random Forest (Train F1): {best_rf_f1_train:.3f}")
print(f"Best Random Forest (Test F1): {best_rf_f1_test:.3f}")

For reference, train a logistic regression model.

In [ ]:
# Initialize Logistic Regression model
log_reg = LogisticRegression(random_state=42, solver='liblinear')

# Fit the model
log_reg.fit(X_train, y_train)

# Predict on training and testing data
log_reg_pred_train = log_reg.predict(X_train)
log_reg_pred_test = log_reg.predict(X_test)

# Calculate F1 scores
log_reg_f1_train = f1_score(y_train, log_reg_pred_train)
log_reg_f1_test = f1_score(y_test, log_reg_pred_test)

print(f"Logistic Regression (Train F1): {log_reg_f1_train:.3f}")
print(f"Logistic Regression (Test F1): {log_reg_f1_test:.3f}")

## Log-Loss Gradient Boosting (AKA *XGBoost*)

Unlike standard decision trees that output class labels or probabilities directly, the
regression trees in a Gradient Boosting Classifier output **log-odds**. The model
builds an additive ensemble of these trees to iteratively refine the log-odds, which
are then converted back into probabilities using the sigmoid function.

### 1. Loss Function
For binary classification where $y_i \in \{0, 1\}$, the algorithm minimizes the
**negative binomial log-likelihood** (often just called log-loss).

If we define $p$ as the predicted probability of the positive class, the standard
log-loss is:
$$
L(y, p) = -y \log(p) - (1-y)\log(1-p)
$$

Because our algorithm predicts the log-odds
$F(x) = \log\left(\frac{p}{1-p}\right)$ rather than the probability directly, we
substitute $p = \frac{e^{F(x)}}{1 + e^{F(x)}}$ into the loss function to express
it in terms of our model's output:
$$
L(y, F(x)) = -y F(x) + \log(1 + e^{F(x)})
$$

The goal of the algorithm is to iteratively build a model $F(x)$ that minimizes
this loss.

### 2. First Tree
Before building any trees, the algorithm makes a single, constant baseline
prediction $F_0(x)$ for all samples. It finds the constant value $\gamma$ that
minimizes the loss function across the entire training set.

Mathematically, setting the derivative of the loss function to zero yields the
global log-odds of the positive class:
$$
F_0(x) = \log \left( \frac{\text{Count of positive samples}}
{\text{Count of negative samples}} \right)
$$

If your dataset is perfectly balanced, $F_0(x)$ will be $\log(1) = 0$, which
corresponds to a probability of **50%**.

### 3. Fitting the First Tree
Now, the algorithm begins its iterative stage. The first tree acts to correct
the mistakes of the baseline model $F_0(x)$.

**A. Compute the Pseudo-Residuals**
Gradient boosting works by fitting regression trees to the negative gradient of
the loss function, which we call "pseudo-residuals." By taking the derivative of
$L(y, F(x))$ with respect to $F(x)$, the pseudo-residual $r_{i1}$ for each
observation $i$ simplifies beautifully to:
$$
r_{i1} = y_i - p_{i0}
$$
Here, $y_i$ is the actual label (0 or 1), and $p_{i0}$ is the probability
predicted by the previous model (converted from $F_0(x)$).

**B. Fit a Regression Tree**
We build a standard regression tree to predict these residuals $r_{i1}$ using
the input features $x_i$. This tree divides the data into $J$ terminal nodes
(leaves), which we can call $R_{j1}$.

**C. Calculate the Leaf Outputs**
Because our tree was built on residuals (which are in the probability space) but
our model needs to output log-odds, we can't just take the average of the
residuals in each leaf. Instead, we use a single Newton-Raphson approximation
step to find the optimal log-odds output $\gamma_{j1}$ for each leaf $j$:
$$
\gamma_{j1} = \frac{\sum_{x_i \in R_{j1}} r_{i1}}
{\sum_{x_i \in R_{j1}} p_{i0}(1 - p_{i0})}
$$
The numerator is the sum of the residuals in the leaf, and the denominator acts
as a scaling factor based on the variance of the previous predictions.

**D. Update the Model**
We update our model by adding the new tree's predictions, scaled by a learning
rate $\nu$ (where $0 < \nu \le 1$):
$$
F_1(x) = F_0(x) + \nu \sum_{j=1}^{J} \gamma_{j1} I(x \in R_{j1})
$$

### 4. Fitting the Second Tree
The process simply loops. The second tree is tasked with correcting the mistakes
of $F_1(x)$.

1. **Update Probabilities:** First, we calculate the new probabilities for every
   sample using the updated model:
   $$
   p_{i1} = \frac{1}{1 + e^{-F_1(x_i)}}
   $$
2. **Compute New Residuals:** We calculate the new pseudo-residuals based on the
   updated probabilities:
   $$
   r_{i2} = y_i - p_{i1}
   $$
3. **Fit Another Tree:** We fit a second regression tree to $r_{i2}$, resulting
   in a new set of leaves $R_{j2}$.
4. **Calculate Leaf Outputs:** We calculate the output values $\gamma_{j2}$ for
   the new leaves, using the updated probabilities in the denominator:
   $$
   \gamma_{j2} = \frac{\sum_{x_i \in R_{j2}} r_{i2}}
   {\sum_{x_i \in R_{j2}} p_{i1}(1 - p_{i1})}
   $$
5. **Update the Model Again:**
   $$
   F_2(x) = F_1(x) + \nu \sum_{j=1}^{J} \gamma_{j2} I(x \in R_{j2})
   $$

This process repeats for $M$ iterations. The final prediction for any new data
point is the sum of the initial log-odds and the scaled log-odds outputs from
all $M$ trees, which is then passed through the logistic function to get the
final probability.

In [ ]:
gbc = GradientBoostingClassifier(
    loss='log_loss',  # Loss function to be optimized
    learning_rate=0.1,  # Step size shrinkage
    n_estimators=100,  # Number of boosting stages
    subsample=1.0,  # Fraction of samples for fitting the individual base learners
    max_depth=3,  # Maximum depth of the individual regression estimators
    random_state=42,  # For reproducibility
    max_features=None,  # Number of features to consider when looking for the best split
)

gbc.fit(X_train, y_train)

gbc_f1_train = f1_score(y_train, gbc.predict(X_train))
gbc_f1_test = f1_score(y_test, gbc.predict(X_test))

print(' Gradient Boosting Classifier '.center(88, '-'))
print(f"F1 train: {gbc_f1_train:.3f}")
print(f"F1 test: {gbc_f1_test:.3f}")

As you can see, we have a serious overfitting problem. This is a textbook example of how
bias decreases and variance increases as model complexity increases.

Let's do a Grid Search to find the best gradient boosting ensemble.

In [ ]:
grid_gbc = {
    'n_estimators': [25, 50, 75, 100],
    'max_depth': [1, 2, 3],
    'max_features': [None, 'sqrt']
}

# Init grid search object for Gradient Boosting Classifier with K=4
search_gbc = GridSearchCV(
    estimator=gbc,
    param_grid=grid_gbc,
    scoring='f1',
    n_jobs=-1,
    cv=4,
    return_train_score=True, refit=True
)

# Fit grid search object
search_gbc.fit(X_train, y_train)

# Store results
search_gbc_results = pd.DataFrame(search_gbc.cv_results_)

# View top models with specified columns
print('Top Gradient Boosting Classifier models:')
display(search_gbc_results[[
    'param_n_estimators', 'param_max_depth', 'param_max_features',
    'mean_train_score', 'mean_test_score', 'rank_test_score'
]].sort_values('rank_test_score', ascending=True).head())

# Get the best estimator
best_gbc_clf = search_gbc.best_estimator_

# Calculate F1 scores for the best Gradient Boosting Classifier
best_gbc_f1_train = f1_score(y_train, best_gbc_clf.predict(X_train))
best_gbc_f1_test = f1_score(y_test, best_gbc_clf.predict(X_test))

print(f"Best Gradient Boosting Classifier (Train F1): {best_gbc_f1_train:.3f}")
print(f"Best Gradient Boosting Classifier (Test F1): {best_gbc_f1_test:.3f}")

Let's compare all models.

In [ ]:
print('\n--- Comparison of Best Models (Test F1 Scores) ---')
print(f"Best Decision Tree (Test F1): {best_dt_f1_test:.3f}")
print(f"Best Random Forest (Test F1): {best_rf_f1_test:.3f}")
print(f"Best Gradient Boosting Classifier (Test F1): {best_gbc_f1_test:.3f}")